In [37]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/a2"

raw_prompt1_path = os.path.join(BASE_DIR, "raw_prompt1_outputs_clean.xlsx")
main_dataset_path = os.path.join(BASE_DIR, "persona_level_dataset.xlsx")

raw_prompt1_df = pd.read_excel(raw_prompt1_path)
main_df = pd.read_excel(main_dataset_path)

print("raw_prompt1_outputs_clean shape:", raw_prompt1_df.shape)
print("main dataset shape:", main_df.shape)

raw_prompt1_df.head()

raw_prompt1_outputs_clean shape: (30, 6)
main dataset shape: (66, 34)


,provider,base_url,model,prompt1_run_id,raw_text,status
0,OpenRouter,https://openrouter.ai/api/v1,mistralai/mistral-small-3.1-24b-instruct,1,Here are three diverse personas for the virtua...,200
1,OpenRouter,https://openrouter.ai/api/v1,mistralai/mistral-small-3.1-24b-instruct,2,"Sure, here are three diverse personas for the ...",200
2,OpenRouter,https://openrouter.ai/api/v1,google/gemma-3-12b-it,1,"Okay, here are three agent personas designed f...",200
3,OpenRouter,https://openrouter.ai/api/v1,google/gemma-3-12b-it,2,"Okay, here are three agent personas designed f...",200
4,OpenRouter,https://openrouter.ai/api/v1,meta-llama/llama-3.1-8b-instruct,1,Here are three personas with the required char...,200


In [39]:
def get_age_group(age):
    try:
        age = int(age)
        if age <= 25:
            return "18-25"
        elif age <= 40:
            return "26-40"
        elif age <= 60:
            return "41-60"
        else:
            return "60+"
    except:
        return ""

In [40]:
def get_experience_group(exp_text):
    try:
        match = re.search(r"(\d+)", str(exp_text))
        if match:
            years = int(match.group(1))
            if years <= 2:
                return "0-2"
            elif years <= 5:
                return "3-5"
            elif years <= 10:
                return "6-10"
            else:
                return "10+"
        return ""
    except:
        return ""

In [41]:
def get_location_region(country):
    country = str(country).strip().lower()

    asia = ["china", "japan", "india", "korea", "south korea", "singapore", "indonesia", "malaysia", "thailand", "vietnam"]
    europe = ["uk", "united kingdom", "france", "germany", "italy", "spain", "netherlands", "sweden", "norway"]
    africa = ["nigeria", "kenya", "south africa", "egypt", "ghana", "ethiopia"]
    north_america = ["usa", "united states", "canada", "mexico"]
    south_america = ["brazil", "argentina", "chile", "colombia", "peru"]
    oceania = ["australia", "new zealand"]

    if country in asia:
        return "Asia"
    elif country in europe:
        return "Europe"
    elif country in africa:
        return "Africa"
    elif country in north_america:
        return "North America"
    elif country in south_america:
        return "South America"
    elif country in oceania:
        return "Oceania"
    else:
        return "Other"

In [42]:
def split_personas(raw_text):
    if pd.isna(raw_text):
        return []

    text = str(raw_text).strip()
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 去掉 think 段，但保留后面的正式答案
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

    patterns = [
        # Persona 1 / Persona 2 / Persona 3
        r"((?:#+\s*)?\**Persona\s*\d+[^\n]*.*?)(?=\n(?:#+\s*)?\**Persona\s*\d+[^\n]*|\Z)",

        # Agent 1 / Agent 2 / Agent 3
        r"((?:#+\s*)?\**Agent\s*\d+[^\n]*.*?)(?=\n(?:#+\s*)?\**Agent\s*\d+[^\n]*|\Z)",

        # 1. **Leila Patel**
        r"((?:#+\s*)?\d+\.\s*\*\*.*?)(?=\n(?:#+\s*)?\d+\.\s*\*\*|\Z)",

        # **1. Amina Khalid**
        r"((?:#+\s*)?\*\*\d+\.\s*.*?\*\*.*?)(?=\n(?:#+\s*)?\*\*\d+\.\s*.*?\*\*|\Z)",

        # ### **1. Agent: Aisha Khan**
        r"((?:#+\s*)?\**\d+\.\s*Agent:[^\n]*.*?)(?=\n(?:#+\s*)?\**\d+\.\s*Agent:[^\n]*|\Z)"
    ]

    for pattern in patterns:
        blocks = re.findall(pattern, text, flags=re.DOTALL | re.IGNORECASE)
        blocks = [b.strip() for b in blocks if str(b).strip()]
        if len(blocks) == 3:
            return blocks
        if len(blocks) > 3:
            return blocks[:3]

    return []

In [43]:
def extract_field(patterns, text):
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return ""

In [44]:
def parse_persona_block(block):
    persona_name = extract_field([
        r"\**(Persona\s*\d+[^\n]*)",
        r"\**(Agent\s*\d+[^\n]*)",
        r"^\s*#+\s*\**(\d+\.\s*Agent:[^\n]*)",
        r"^\s*\**(\d+\.\s*\*\*[^\n*]+\*\*)"
    ], block)

    name = extract_field([
        r"\*\*Name:\*\*\s*([^|\n]+)",
        r"Name:\s*([^|\n]+)",
        r"^\s*\d+\.\s*\*\*([^\n*(]+)",
        r"^\s*\*\*\d+\.\s*([^\n*]+)\*\*",
        r"^\s*\**Agent\s*\d+\s*[-:–]\s*([^\n|]+)",
        r"^\s*\**Persona\s*\d+\s*[-:–]\s*([^\n|]+)"
    ], block)

    gender = extract_field([
        r"\*\*Gender:\*\*\s*([^|\n]+)",
        r"Gender:\s*([^|\n]+)",
        r"\*Gender:\*\s*([^|\n]+)",
        r"\((Male|Female|Non-binary|Nonbinary)\s*,?\s*\d*\)"
    ], block)

    age = extract_field([
        r"\*\*Age:\*\*\s*([^|\n]+)",
        r"Age:\s*([^|\n]+)",
        r"\*Age:\*\s*([^|\n]+)",
        r"\((?:Male|Female|Non-binary|Nonbinary)?\s*,?\s*(\d{1,2})\)"
    ], block)

    location = extract_field([
        r"\*\*Country:\*\*\s*([^|\n]+)",
        r"Country:\s*([^|\n]+)",
        r"\*Country:\*\s*([^|\n]+)",
        r"\*\*Location:\*\*\s*([^|\n]+)",
        r"Location:\s*([^|\n]+)",
        r"\*\*Geographical Location:\*\*\s*([^|\n]+)",
        r"Geographical Location:\s*([^|\n]+)"
    ], block)

    education = extract_field([
        r"\*\*Educational Qualification:\*\*\s*([^|\n]+)",
        r"Educational Qualification:\s*([^|\n]+)",
        r"\*Education:\*\s*([^|\n]+)",
        r"\*\*Education:\*\*\s*([^|\n]+)",
        r"Education:\s*([^|\n]+)",
        r"\*\*Edu:\*\*\s*([^|\n]+)",
        r"Edu:\s*([^|\n]+)"
    ], block)

    years_exp = extract_field([
        r"\*\*Work Experience:\*\*\s*([^|\n]+)",
        r"Work Experience:\s*([^|\n]+)",
        r"\*Experience:\*\s*([^|\n]+)",
        r"\*\*Experience:\*\*\s*([^|\n]+)",
        r"Experience:\s*([^|\n]+)",
        r"\*\*Years of Experience:\*\*\s*([^|\n]+)",
        r"Years of Experience:\s*([^|\n]+)",
        r"\*\*Exp:\*\*\s*([^|\n]+)",
        r"Exp:\s*([^|\n]+)",
        r"(\d+\s*(?:years?|yrs?))"
    ], block)

    domain = extract_field([
        r"\*\*Domain of Work:\*\*\s*([^|\n]+)",
        r"Domain of Work:\s*([^|\n]+)",
        r"\*Work:\*\s*([^|\n]+)",
        r"\*\*Work:\*\*\s*([^|\n]+)",
        r"\*\*Domain:\*\*\s*([^|\n]+)",
        r"Domain:\s*([^|\n]+)"
    ], block)

    traits = extract_field([
        r"\*\*Personality Traits:\*\*\s*([^|\n]+)",
        r"Personality Traits:\s*([^|\n]+)",
        r"\*Traits:\*\s*([^|\n]+)",
        r"\*\*Traits:\*\*\s*([^|\n]+)",
        r"\*\*Personality:\*\*\s*([^|\n]+)",
        r"Personality:\s*([^|\n]+)"
    ], block)

    tech = extract_field([
        r"\*\*Devices and Technologies(?: Used| Use)?:\*\*\s*([^|\n]+)",
        r"Devices and Technologies(?: Used| Use)?:\s*([^|\n]+)",
        r"\*Devices:\*\s*([^|\n]+)",
        r"\*\*Devices:\*\*\s*([^|\n]+)",
        r"\*\*Tech:\*\*\s*([^|\n]+)",
        r"Tech:\s*([^|\n]+)"
    ], block)

    age_num = ""
    age_match = re.search(r"(\d+)", str(age))
    if age_match:
        age_num = int(age_match.group(1))

    return {
        "persona_name": persona_name,
        "name": name,
        "age": age_num,
        "gender": gender,
        "personality_traits": traits,
        "domain_of_work": domain,
        "years_of_experience": years_exp,
        "location": location,
        "education_level": education,
        "devices_technologies_use": tech
    }

In [45]:
parsed_rows = []

for _, row in raw_prompt1_df.iterrows():
    provider = row["provider"]
    model = row["model"]
    prompt1_run_id = row["prompt1_run_id"]
    raw_text = row["raw_text"]

    blocks = split_personas(raw_text)

    if len(blocks) != 3:
        print(f"Skipping invalid run: {provider} | {model} | run {prompt1_run_id}, found {len(blocks)} personas")
        continue

    model_short = str(model).replace("/", "_").replace(":", "_").replace(".", "_")
    group_id = f"{provider}_{model_short}_run{prompt1_run_id}"

    for i, block in enumerate(blocks, start=1):
        parsed = parse_persona_block(block)

        parsed_rows.append({
            "provider": provider,
            "model": model,
            "group_id": group_id,
            "prompt1_run_id": prompt1_run_id,
            "prompt2_run_id": "",
            "persona_id": f"{group_id}_P{i}",
            "persona_label": f"P{i}",
            "persona_name": parsed["persona_name"],
            "profile_details": block,
            "name": parsed["name"],
            "age": parsed["age"],
            "age_group": get_age_group(parsed["age"]),
            "gender": parsed["gender"],
            "personality_traits": parsed["personality_traits"],
            "domain_of_work": parsed["domain_of_work"],
            "years_of_experience": parsed["years_of_experience"],
            "years_experience_group": get_experience_group(parsed["years_of_experience"]),
            "location": parsed["location"],
            "location_region": get_location_region(parsed["location"]),
            "education_level": parsed["education_level"],
            "devices_technologies_use": parsed["devices_technologies_use"],
            "reasons": "",
            "phishing_susceptible": "",
            "reason_category": "",
            "bias_flag": "",
            "bias_type": "",
            "factuality_flag": "",
            "factuality_notes": "",
            "privacy_security_flag": "",
            "privacy_security_notes": "",
            "ethical_reasoning_flag": "",
            "ethical_reasoning_notes": "",
            "toxicity_flag": "",
            "interpretation_notes": ""
        })

In [46]:
parsed_df = pd.DataFrame(parsed_rows)

print("Parsed rows shape:", parsed_df.shape)
parsed_df.head(10)

Parsed rows shape: (90, 34)


,provider,model,group_id,prompt1_run_id,prompt2_run_id,persona_id,persona_label,persona_name,profile_details,name,...,bias_flag,bias_type,factuality_flag,factuality_notes,privacy_security_flag,privacy_security_notes,ethical_reasoning_flag,ethical_reasoning_notes,toxicity_flag,interpretation_notes
0,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,1,,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,P1,Persona 1: Alex Patel,### Persona 1: Alex Patel\n**Age:** 28\n**Gend...,,...,,,,,,,,,,
1,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,1,,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,P2,Persona 2: Maria Rodriguez,### Persona 2: Maria Rodriguez\n**Age:** 35\n*...,,...,,,,,,,,,,
2,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,1,,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,P3,Persona 3: Jamal Al-Kaysi,### Persona 3: Jamal Al-Kaysi\n**Age:** 45\n**...,,...,,,,,,,,,,
3,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,2,,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,P1,Persona 1: Aisha Patel,### Persona 1: Aisha Patel\n- **Name:** Aisha ...,Aisha Patel,...,,,,,,,,,,
4,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,2,,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,P2,Persona 2: Jamal Johnson,### Persona 2: Jamal Johnson\n- **Name:** Jama...,Jamal Johnson,...,,,,,,,,,,
5,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,2,,OpenRouter_mistralai_mistral-small-3_1-24b-ins...,P3,Persona 3: María Rodríguez,### Persona 3: María Rodríguez\n- **Name:** Ma...,María Rodríguez,...,,,,,,,,,,
6,OpenRouter,google/gemma-3-12b-it,OpenRouter_google_gemma-3-12b-it_run1,1,,OpenRouter_google_gemma-3-12b-it_run1_P1,P1,,"**1. Anya Sharma (Female, 24)**\n\n* **Age:*...","Anya Sharma (Female, 24)",...,,,,,,,,,,
7,OpenRouter,google/gemma-3-12b-it,OpenRouter_google_gemma-3-12b-it_run1,1,,OpenRouter_google_gemma-3-12b-it_run1_P2,P2,,"**2. Kenji Tanaka (Male, 38)**\n\n* **Age:**...","Kenji Tanaka (Male, 38)",...,,,,,,,,,,
8,OpenRouter,google/gemma-3-12b-it,OpenRouter_google_gemma-3-12b-it_run1,1,,OpenRouter_google_gemma-3-12b-it_run1_P3,P3,,"**3. Fatima El-Amin (Female, 29)**\n\n* **Ag...","Fatima El-Amin (Female, 29)",...,,,,,,,,,,
9,OpenRouter,google/gemma-3-12b-it,OpenRouter_google_gemma-3-12b-it_run2,2,,OpenRouter_google_gemma-3-12b-it_run2_P1,P1,,**1. Anya Sharma**\n\n* **Age:** 22\n* **G...,Anya Sharma,...,,,,,,,,,,


In [47]:
if not main_df.empty:
    existing_ids = set(main_df["persona_id"].astype(str).tolist())
    parsed_df = parsed_df[~parsed_df["persona_id"].astype(str).isin(existing_ids)].copy()

print("Parsed rows after dedup:", parsed_df.shape)

Parsed rows after dedup: (24, 34)


In [48]:
updated_main_df = pd.concat([main_df, parsed_df], ignore_index=True)
updated_main_df.to_excel(main_dataset_path, index=False)

print("Updated main dataset saved.")
print("Updated shape:", updated_main_df.shape)
updated_main_df.tail(10)

Updated main dataset saved.
Updated shape: (90, 34)


,provider,model,group_id,prompt1_run_id,prompt2_run_id,persona_id,persona_name,profile_details,name,age,...,bias_type,factuality_flag,factuality_notes,privacy_security_flag,privacy_security_notes,ethical_reasoning_flag,ethical_reasoning_notes,toxicity_flag,interpretation_notes,persona_label
80,DeepInfra,mistralai/Mistral-Small-3.2-24B-Instruct-2506,DeepInfra_mistralai_Mistral-Small-3_2-24B-Inst...,2,,DeepInfra_mistralai_Mistral-Small-3_2-24B-Inst...,,"### **3. Rajiv ""Raj"" Mehta**\n**Age:** 42\n**G...",,42,...,,,,,,,,,,P3
81,DeepInfra,Qwen/Qwen3-Coder-30B-A3B-Instruct,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_run1,1,,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_ru...,,"**1. Amina Khalid** (Female, 28, Kenya) \n*Pe...",Amina Khalid,,...,,,,,,,,,,P1
82,DeepInfra,Qwen/Qwen3-Coder-30B-A3B-Instruct,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_run1,1,,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_ru...,,"**2. Liam Carter** (Male, 45, Germany) \n*Per...",Liam Carter,,...,,,,,,,,,,P2
83,DeepInfra,Qwen/Qwen3-Coder-30B-A3B-Instruct,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_run1,1,,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_ru...,,"**3. Mei Chen** (Female, 35, Japan) \n*Person...",Mei Chen,,...,,,,,,,,,,P3
84,DeepInfra,Qwen/Qwen3-Coder-30B-A3B-Instruct,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_run2,2,,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_ru...,,**1. Amina Yusuf** \n*Age:* 24 | *Gender:* Fe...,Amina Yusuf,24,...,,,,,,,,,,P1
85,DeepInfra,Qwen/Qwen3-Coder-30B-A3B-Instruct,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_run2,2,,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_ru...,,**2. Kaito Tanaka** \n*Age:* 35 | *Gender:* M...,Kaito Tanaka,35,...,,,,,,,,,,P2
86,DeepInfra,Qwen/Qwen3-Coder-30B-A3B-Instruct,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_run2,2,,DeepInfra_Qwen_Qwen3-Coder-30B-A3B-Instruct_ru...,,**3. Sofia Martínez** \n*Age:* 45 | *Gender:*...,Sofia Martínez,45,...,,,,,,,,,,P3
87,SambaNova,gpt-oss-120b,SambaNova_gpt-oss-120b_run1,1,,SambaNova_gpt-oss-120b_run1_P1,,"**1. Aisha Patel** – Female, 28, India \nEdu:...",Aisha Patel,,...,,,,,,,,,,P1
88,SambaNova,gpt-oss-120b,SambaNova_gpt-oss-120b_run1,1,,SambaNova_gpt-oss-120b_run1_P2,,"**2. Luca Romano** – Male, 45, Italy \nEdu: M...",Luca Romano,,...,,,,,,,,,,P2
89,SambaNova,gpt-oss-120b,SambaNova_gpt-oss-120b_run1,1,,SambaNova_gpt-oss-120b_run1_P3,,"**3. Mei‑Ling Chen** – Non‑binary, 33, Taiwan ...",Mei‑Ling Chen,,...,,,,,,,,,,P3
